# Step H — CO₂ Shadow Price Analysis

**Model:** The Step D interconnected system (DK extendable, DE/SE/NO fixed).

### Emission factors used (IPCC 2006, Table 2.2)
| Carrier  | ε [tCO₂/MWh_th] | Source |
|----------|-----------------|--------|
| CCGT (gas) | 0.202         | IPCC 2006 |
| Coal       | 0.341         | IPCC 2006 |



## Imports

In [ ]:
from pathlib import Path
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## H0 — Rebuild the Step D network

This block reproduces the full Step D setup exactly.
The key addition here vs. Step D is that **CO₂ emission factors are assigned
to the CCGT and coal carriers** so PyPSA can enforce a GlobalConstraint.

**Emission factors (IPCC 2006, Table 2.2):**
- Natural gas (CCGT): ε = 0.202 tCO₂/MWh_thermal
- Hard coal:          ε = 0.341 tCO₂/MWh_thermal

PyPSA internally computes the electrical-equivalent emission as ε/η,
so we only need to supply the thermal factor and the generator efficiency.


In [ ]:
# ── CELL H0a — paths and data loading ───────────────────────
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "STEP D - electricity_demand.csv").exists():
    PROJECT_DIR = PROJECT_DIR.parent

# --- Denmark data ---
dataframe_dk = pd.read_csv(
    PROJECT_DIR / "DK_2015_merged.csv",
    index_col=0, sep=",", parse_dates=True
)
dataframe_dk.index = pd.to_datetime(dataframe_dk.index, utc=True).tz_localize(None)
dataframe_dk = dataframe_dk.sort_index()

CF_wind  = dataframe_dk["wind_cf_Unnamed: 1"].astype(float)
CF_solar = dataframe_dk["pv_cf_Unnamed: 1"].astype(float)
snapshots = dataframe_dk.index

# --- Multi-country demand ---
demand_all = pd.read_csv(
    PROJECT_DIR / "STEP D - electricity_demand.csv",
    sep=";", index_col=0, parse_dates=True
)
demand_all.index = pd.to_datetime(demand_all.index, utc=True).tz_localize(None)
demand_all = demand_all.sort_index()
demand_2015 = demand_all[demand_all.index.year == 2015].copy()

demand_dk = demand_2015["DNK"].astype(float).reindex(snapshots)
demand_de = demand_2015["DEU"].astype(float).reindex(snapshots)
demand_se = demand_2015["SWE"].astype(float).reindex(snapshots)
demand_no = demand_2015["NOR"].astype(float).reindex(snapshots)

# --- Neighbouring country capacity factors ---
cf_base = PROJECT_DIR / "Other country data" / "Energy charts data"

cf_de_raw = pd.read_csv(cf_base / "Germany" / "Germany_hourly_capacity_factors.csv")
cf_de_raw["timestamp"] = pd.to_datetime(cf_de_raw["timestamp"])
cf_de_raw = cf_de_raw.set_index("timestamp").sort_index()
cf_de_raw.index = cf_de_raw.index.tz_localize(None)
cf_de = cf_de_raw.resample("h").mean()
cf_de["wind_combined"] = cf_de["Wind onshore"]
cf_de["solar"]         = cf_de["Solar AC"]
cf_de["CCGT"]          = cf_de["Fossil gas"]
cf_de = cf_de[["wind_combined", "solar", "CCGT"]].reindex(snapshots)

cf_se = pd.read_csv(cf_base / "Sweden" / "Sweden_hourly_capacity_factors.csv")
cf_se["timestamp"] = pd.to_datetime(cf_se["timestamp"])
cf_se = cf_se.set_index("timestamp").sort_index()
cf_se.index = cf_se.index.tz_localize(None)
cf_se["wind_combined"] = cf_se["Wind onshore"]
cf_se["nuclear"]       = cf_se["Nuclear"]
cf_se["hydro"]         = cf_se["Hydro water reservoir"]
cf_se = cf_se[["wind_combined", "nuclear", "hydro"]].reindex(snapshots).bfill()

cf_no = pd.read_csv(cf_base / "Norway" / "Norway_hourly_capacity_factors.csv")
cf_no["timestamp"] = pd.to_datetime(cf_no["timestamp"])
cf_no = cf_no.set_index("timestamp").sort_index()
cf_no.index = cf_no.index.tz_localize(None)
cf_no["wind_combined"] = cf_no["Wind onshore"]
cf_no["hydro"]         = cf_no["Hydro water reservoir"]
cf_no = cf_no[["wind_combined", "hydro"]].reindex(snapshots)

print("Data loaded successfully.")
print(f"  Snapshots: {len(snapshots)} hours ({snapshots[0]} → {snapshots[-1]})")

In [ ]:
# ── CELL H0b — cost table (identical to Step D / Step A) ────
data = {
    "capital_cost": [
        1500000/25 + 60000,   # wind:  120,000 $/MW/year
        800000/25  + 14000,   # solar:  46,000 $/MW/year
        700000/25  + 24000,   # CCGT:   52,000 $/MW/year
    ],
    "marginal_cost": [0.0, 0.0, 9.5 * 3.6 / 0.56 + 2.30],  # CCGT: ~63.4 $/MWh
}
costs = pd.DataFrame(data, index=["wind_combined", "solar", "CCGT"])

# Battery parameters (2024 Li-ion, same as Step D)
battery_investment_power  = 100_000   # $/MW
battery_investment_energy = 150_000   # $/MWh
battery_fom               =  12_500   # $/MW/year
battery_lifetime          =      20   # years
battery_max_hours         =       4   # h
battery_efficiency        =    0.90   # round-trip
battery_capital_cost = (
    battery_investment_power  / battery_lifetime
    + battery_fom
    + (battery_investment_energy / battery_lifetime) * battery_max_hours
)

print(f"CCGT capital cost:    {costs.loc['CCGT','capital_cost']:,.0f} $/MW/year")
print(f"Battery capital cost: {battery_capital_cost:,.0f} $/MW/year")

In [ ]:
# ── CELL H0c — network builder function ─────────────────────
#
# Wrapping the network construction in a function lets us rebuild
# cleanly for the constrained run without carrying stale state.
#
# KEY ADDITION vs Step D:
#   co2_emissions=[0, 0, 0.202, 0, 0, 0.341, 0] on the Carrier add.
#   This tells PyPSA the thermal CO₂ factor per carrier.
#   PyPSA then computes the effective electrical factor as ε/η internally.

# IPCC 2006 CO₂ emission factors [tCO₂/MWh_thermal]
CO2_GAS  = 0.202   # natural gas (Table 2.2, IPCC 2006)
CO2_COAL = 0.341   # hard coal   (Table 2.2, IPCC 2006)

def build_network_h(co2_limit=None):
    """
    Build the Step D interconnected network.
    If co2_limit is given (in tCO₂/year), a GlobalConstraint is added.
    Returns the optimised network.
    """
    n = pypsa.Network()
    n.set_snapshots(snapshots)

    # ── Buses ────────────────────────────────────────────────
    buses = {
        "Denmark": (10.0, 56.0),
        "Germany": (10.5, 51.5),
        "Sweden":  (15.0, 59.5),
        "Norway":  (10.0, 62.0),
    }
    for name, (x, y) in buses.items():
        n.add("Bus", name, x=x, y=y, v_nom=380)

    # ── Carriers — with CO₂ emission factors ─────────────────
    # Order: wind_onshore, solar, CCGT, hydro, nuclear, coal, battery
    n.add(
        "Carrier",
        ["wind_onshore", "solar", "CCGT", "hydro", "nuclear", "coal", "battery"],
        color=["blue", "yellow", "brown", "cyan", "purple", "grey", "violet"],
        co2_emissions=[0, 0, CO2_GAS, 0, 0, CO2_COAL, 0],
    )

    # ── Loads ────────────────────────────────────────────────
    n.add("Load", "load_DK", bus="Denmark", p_set=demand_dk.values)
    n.add("Load", "load_DE", bus="Germany", p_set=demand_de.values)
    n.add("Load", "load_SE", bus="Sweden",  p_set=demand_se.values)
    n.add("Load", "load_NO", bus="Norway",  p_set=demand_no.values)

    # ── Denmark generators (extendable) ──────────────────────
    n.add("Generator", "DK_wind",
          bus="Denmark", carrier="wind_onshore",
          capital_cost=costs.loc["wind_combined", "capital_cost"],
          marginal_cost=costs.loc["wind_combined", "marginal_cost"],
          p_max_pu=CF_wind.values, p_nom_extendable=True)

    n.add("Generator", "DK_solar",
          bus="Denmark", carrier="solar",
          capital_cost=costs.loc["solar", "capital_cost"],
          marginal_cost=costs.loc["solar", "marginal_cost"],
          p_max_pu=CF_solar.values, p_nom_extendable=True)

    n.add("Generator", "DK_CCGT",
          bus="Denmark", carrier="CCGT",
          capital_cost=costs.loc["CCGT", "capital_cost"],
          marginal_cost=costs.loc["CCGT", "marginal_cost"],
          efficiency=0.58, p_nom_extendable=True)

    # ── Germany generators (fixed) ───────────────────────────
    n.add("Generator", "DE_wind",
          bus="Germany", carrier="wind_onshore",
          p_nom=41300, marginal_cost=0,
          p_max_pu=cf_de["wind_combined"].values, p_nom_extendable=False)

    n.add("Generator", "DE_solar",
          bus="Germany", carrier="solar",
          p_nom=37000, marginal_cost=0,
          p_max_pu=cf_de["solar"].values, p_nom_extendable=False)

    n.add("Generator", "DE_CCGT",
          bus="Germany", carrier="CCGT",
          p_nom=28360, marginal_cost=62.0,
          efficiency=0.56, p_nom_extendable=False)

    n.add("Generator", "DE_nuclear",
          bus="Germany", carrier="nuclear",
          p_nom=10800, marginal_cost=11.0, p_nom_extendable=False)

    n.add("Generator", "DE_coal",
          bus="Germany", carrier="coal",
          p_nom=21420, marginal_cost=40.0,
          efficiency=0.40, p_nom_extendable=False)

    # ── Sweden generators (fixed) ────────────────────────────
    n.add("Generator", "SE_hydro",
          bus="Sweden", carrier="hydro",
          p_nom=15920, marginal_cost=5.0,
          p_max_pu=cf_se["hydro"].values, p_nom_extendable=False)

    n.add("Generator", "SE_nuclear",
          bus="Sweden", carrier="nuclear",
          p_nom=8900, marginal_cost=11.0, p_nom_extendable=False)

    n.add("Generator", "SE_wind",
          bus="Sweden", carrier="wind_onshore",
          p_nom=5500, marginal_cost=0,
          p_max_pu=cf_se["wind_combined"].values, p_nom_extendable=False)

    # ── Norway generators (fixed) ────────────────────────────
    n.add("Generator", "NO_hydro",
          bus="Norway", carrier="hydro",
          p_nom=29900, marginal_cost=5.0,
          p_max_pu=cf_no["hydro"].values, p_nom_extendable=False)

    n.add("Generator", "NO_wind",
          bus="Norway", carrier="wind_onshore",
          p_nom=700, marginal_cost=0,
          p_max_pu=cf_no["wind_combined"].values, p_nom_extendable=False)

    # ── Batteries (extendable at all buses) ──────────────────
    for country, bus in [("DK","Denmark"),("DE","Germany"),("SE","Sweden"),("NO","Norway")]:
        n.add("StorageUnit", f"{country}_battery",
              bus=bus, carrier="battery",
              capital_cost=battery_capital_cost,
              marginal_cost=0,
              efficiency_store=battery_efficiency**0.5,
              efficiency_dispatch=battery_efficiency**0.5,
              max_hours=battery_max_hours,
              cyclic_state_of_charge=True,
              p_nom_extendable=True)

    # ── Transmission lines ───────────────────────────────────
    lines = [
        ("line_DK_DE", "Denmark", "Germany", 2380, 350),
        ("line_DK_SE", "Denmark", "Sweden",  2000, 180),
        ("line_DK_NO", "Denmark", "Norway",  1680, 300),
        ("line_SE_NO", "Sweden",  "Norway",  2100, 500),
        ("line_DE_SE", "Germany", "Sweden",   600, 700),
    ]
    for name, bus0, bus1, s_nom, length_km in lines:
        n.add("Line", name,
              bus0=bus0, bus1=bus1,
              s_nom=s_nom, x=0.1, r=0.01, length=length_km,
              p_nom=s_nom)

    # ── Optional CO₂ GlobalConstraint ────────────────────────
    if co2_limit is not None:
        n.add(
            "GlobalConstraint",
            "co2_limit",
            sense="<=",
            carrier_attribute="co2_emissions",
            constant=co2_limit,   # tCO₂/year
        )

    # ── Optimise ─────────────────────────────────────────────
    status, condition = n.optimize(
        solver_name="gurobi",
        solver_options={"output_flag": False}
    )
    if status != "ok":
        raise RuntimeError(f"Optimisation failed: {status} / {condition}")

    return n

print("Network builder function defined.")

## H1 — Compute baseline (unconstrained) emissions E₀

We run the model **without** any CO₂ constraint to find how much the system
emits when only minimising cost. This is our 2015 model baseline $E_0$.

The emitting generators are:
- **DK_CCGT**: η = 0.58, ε = 0.202 tCO₂/MWh_th → 0.348 tCO₂/MWh_el
- **DE_CCGT**: η = 0.56, ε = 0.202 tCO₂/MWh_th → 0.361 tCO₂/MWh_el
- **DE_coal**: η = 0.40, ε = 0.341 tCO₂/MWh_th → 0.853 tCO₂/MWh_el


In [ ]:
# ── CELL H1 — unconstrained run ──────────────────────────────
print("Running unconstrained optimisation...")
net_base = build_network_h(co2_limit=None)
print(f"Done. Objective: {net_base.objective/1e9:.3f} B$/year")

# --- Manual emission calculation ---
# (Also available via net_base.global_constraints if a constraint was active,
#  but here we compute it explicitly for transparency.)

emitters = {
    "DK_CCGT": {"eps": 0.202, "eta": 0.58},
    "DE_CCGT": {"eps": 0.202, "eta": 0.56},
    "DE_coal": {"eps": 0.341, "eta": 0.40},
}

print("\n--- Baseline CO₂ emissions by generator ---")
total_E0 = 0.0
for gen, params in emitters.items():
    gen_MWh   = net_base.generators_t.p[gen].sum()          # MWh_el/year
    emissions = gen_MWh * params["eps"] / params["eta"]     # tCO₂/year
    total_E0 += emissions
    print(f"  {gen:<10}: {gen_MWh/1e6:.2f} TWh_el/y  →  {emissions/1e6:.3f} MtCO₂/y")

E0_Mt = total_E0 / 1e6   # MtCO₂/year
print(f"\n  TOTAL E₀ = {E0_Mt:.3f} MtCO₂/year")
print(f"  (= {total_E0:,.0f} tCO₂/year)")

## H2 — Choose the decarbonisation target

We target a **50% reduction from the model baseline E₀**, which is broadly
consistent with the EU's Fit for 55 target (55% reduction by 2030 vs. 1990).

Note: our model represents a single optimised year (2015) and does not
capture investment pathways. The cap is a static constraint, not a trajectory.


In [ ]:
# ── CELL H2 — set the CO₂ target ────────────────────────────
REDUCTION = 0.30   # 50% reduction from baseline

co2_target_t = total_E0 * (1 - REDUCTION)   # tCO₂/year  ← passed to PyPSA
co2_target_Mt = co2_target_t / 1e6

print(f"Baseline emissions E₀:  {E0_Mt:.3f} MtCO₂/year")
print(f"Reduction target:       {REDUCTION*100:.0f}%")
print(f"CO₂ cap:                {co2_target_Mt:.3f} MtCO₂/year")
print(f"                        ({co2_target_t:,.0f} tCO₂/year — value passed to PyPSA)")

## H3 — Run the constrained optimisation

We now re-run with the CO₂ GlobalConstraint active.
The **shadow price** (dual variable / Lagrange multiplier) of this constraint
is the marginal cost of reducing emissions by 1 tCO₂ at the optimum —
i.e., the system-wide CO₂ price needed to achieve this decarbonisation level.


In [ ]:
# ── CELL H3 — constrained run ────────────────────────────────
print(f"Running constrained optimisation (cap = {co2_target_Mt:.3f} MtCO₂/year)...")
net_co2 = build_network_h(co2_limit=co2_target_t)
print(f"Done. Objective: {net_co2.objective/1e9:.3f} B$/year")

# --- Extract shadow price ---
# PyPSA stores dual variables of GlobalConstraints in net.global_constraints["mu"]
# The sign convention: mu >= 0 means the constraint is binding and costly.
shadow_price = abs(net_co2.global_constraints.loc["co2_limit", "mu"])
print(f"\nCO₂ shadow price (μ): {shadow_price:.2f} $/tCO₂")

# --- Verify actual emissions under the constrained solution ---
print("\n--- Constrained emissions by generator ---")
total_E_constrained = 0.0
for gen, params in emitters.items():
    gen_MWh   = net_co2.generators_t.p[gen].sum()
    emissions = gen_MWh * params["eps"] / params["eta"]
    total_E_constrained += emissions
    print(f"  {gen:<10}: {gen_MWh/1e6:.2f} TWh_el/y  →  {emissions/1e6:.3f} MtCO₂/y")

print(f"\n  TOTAL constrained = {total_E_constrained/1e6:.3f} MtCO₂/year")
print(f"  Target cap        = {co2_target_Mt:.3f} MtCO₂/year")
print(f"  Constraint binding: {abs(total_E_constrained - co2_target_t) < 1e3}")

## H4 — Compare optimal capacities: baseline vs constrained

How does the CO₂ cap change what gets built in Denmark?


In [ ]:
# ── CELL H4 — capacity comparison ───────────────────────────
dk_gens   = ["DK_wind", "DK_solar", "DK_CCGT"]
dk_batt   = ["DK_battery"]

print("=== Denmark optimal capacities [GW] ===")
print(f"{'Generator':<12} {'Baseline':>10} {'Constrained':>13} {'Change':>10}")
print("-" * 50)
for gen in dk_gens + dk_batt:
    if gen in net_base.generators.index:
        base_val = net_base.generators.loc[gen, "p_nom_opt"]
        co2_val  = net_co2.generators.loc[gen, "p_nom_opt"]
    else:
        base_val = net_base.storage_units.loc[gen, "p_nom_opt"]
        co2_val  = net_co2.storage_units.loc[gen, "p_nom_opt"]
    pct = (co2_val - base_val) / base_val * 100 if base_val > 0 else float("nan")
    print(f"  {gen:<12} {base_val/1e3:>8.2f} GW   {co2_val/1e3:>8.2f} GW   {pct:>+8.1f}%")

print(f"\nSystem cost increase: "
      f"{(net_co2.objective - net_base.objective)/1e6:+.1f} M$/year "
      f"({(net_co2.objective/net_base.objective - 1)*100:+.1f}%)")

## H5 — Compare with real-world CO₂ prices

Reference prices (2015 values unless stated):
| Scheme | Price [$/tCO₂] | Notes |
|--------|---------------|-------|
| EU ETS (2015) | ~7 | Very low due to surplus allowances (source: Ember) |
| EU ETS (2024) | ~60–70 | Post-reform recovery |
| Denmark CO₂ tax (2015) | ~30 | On top of ETS (source: World Bank CPD) |
| Denmark CO₂ tax (2030 target) | ~160 | Danish Climate Act 2020 |
| Sweden carbon tax (2015) | ~120 | One of the world's highest (source: World Bank CPD) |

Source: World Bank Carbon Pricing Dashboard; Ember European Carbon Price Tracker


In [ ]:
# ── CELL H5 — shadow price vs real-world comparison plot ─────
fig, ax = plt.subplots(figsize=(9, 5))

# Real-world reference prices [$/tCO₂]
references = {
    "EU ETS\n(2015)": 7,
    "DK CO₂ tax\n(2015)": 30,
    "EU ETS\n(2024)": 65,
    "DK CO₂ tax\n(2030 target)": 160,
    "SE carbon tax\n(2015)": 120,
}

labels = list(references.keys())
prices = list(references.values())
colors = ["#aec6e8"] * len(labels)

bars = ax.bar(labels, prices, color=colors, edgecolor="white", width=0.5, zorder=2)

# Model shadow price as horizontal line
ax.axhline(shadow_price, color="#d62728", linewidth=2.5, linestyle="--", zorder=3,
           label=f"Model shadow price: {shadow_price:.0f} $/tCO₂")

ax.set_ylabel("CO₂ price [$/tCO₂]", fontsize=12)
ax.set_title(
    f"CO₂ shadow price at {REDUCTION*100:.0f}% emission reduction vs real-world prices",
    fontsize=13
)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=1)
ax.set_ylim(0, max(max(prices), shadow_price) * 1.25)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

plt.tight_layout()
plt.savefig("step_H_shadow_price_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## H6 — Summary and discussion


In [ ]:
# ── CELL H6 — print summary table ───────────────────────────
shadow_price = abs(net_co2.global_constraints.loc["co2_limit", "mu"])
print("="*55)
print("  STEP H — Summary")
print("="*55)
print(f"  Baseline emissions E₀:       {E0_Mt:.3f} MtCO₂/year")
print(f"  CO₂ cap ({REDUCTION*100:.0f}% reduction):     {co2_target_Mt:.3f} MtCO₂/year")
print(f"  Actual constrained emissions: {total_E_constrained/1e6:.3f} MtCO₂/year")
print(f"  CO₂ shadow price (μ):         {shadow_price:.1f} $/tCO₂")
print(f"  System cost increase:         "
      f"{(net_co2.objective - net_base.objective)/1e6:.1f} M$/year")
print("="*55)
print()
print("Real-world reference prices:")
for name, price in references.items():
    marker = "<-- model" if abs(price - shadow_price) == min(
        abs(p - shadow_price) for p in prices) else ""
    print(f"  {name.replace(chr(10),' '):<30} {price:>5} $/tCO₂  {marker}")
print()
print("Key discussion points:")
print("  1. Is the shadow price above or below the 2015 EU ETS price?")
print("     → If above: the EU ETS was too cheap to drive this reduction.")
print("  2. Does it align with any national CO₂ tax?")
print("  3. Why might the model's price differ from real-world prices?")
print("     - Perfect foresight (no uncertainty)")
print("     - Fixed neighbour capacities (no full system response)")
print("     - Only electricity sector modelled (no fuel switching elsewhere)")
print("     - 2020 technology costs used (cheaper wind/solar than 2015 reality)")